# Subsidence & Groundwater (Texas) downloader

This notebook downloads a seed corpus of plans, news articles, technical reports, legal texts, and stakeholder materials for CKAN ingestion.

It is designed to be **defensive**:
- it streams downloads
- infers file extensions from content type where possible
- saves HTML pages when a direct file is not available
- records every success/failure in a manifest CSV

It uses `subsidence_groundwater_seed_urls.csv` in this same folder as the source of truth for download targets.

Downloads are written into the local tutorial data directory at:
`./data/subsidence_groundwater_corpus/`

If a URL has changed, the notebook will log the failure instead of stopping.


In [1]:
import csv
import json
import mimetypes
import re
import time
from pathlib import Path
from urllib.parse import urlparse

from bs4 import BeautifulSoup
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

SEED_FILENAME = "subsidence_groundwater_seed_urls.csv"


def resolve_notebook_dir(start: Path | None = None) -> Path:
    """Find the directory that contains this notebook's companion seed CSV."""
    start = (start or Path.cwd()).resolve()
    search_roots = [start] + list(start.parents)

    for root in search_roots:
        candidate = root / SEED_FILENAME
        if candidate.exists():
            return root

    for root in search_roots:
        candidate = root / "DSO-Institute-2026" / "Day-3" / "Morning" / SEED_FILENAME
        if candidate.exists():
            return candidate.parent

    raise FileNotFoundError(
        f"Could not locate {SEED_FILENAME} from starting directory: {start}"
    )


NOTEBOOK_DIR = resolve_notebook_dir()
SEED_PATH = NOTEBOOK_DIR / SEED_FILENAME
TUTORIAL_DATA_DIR = NOTEBOOK_DIR / "data"
BASE_DIR = TUTORIAL_DATA_DIR / "subsidence_groundwater_corpus"
RAW_DIR = BASE_DIR / "raw"
CLEAN_DIR = BASE_DIR / "cleaned"
META_DIR = BASE_DIR / "metadata"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "Mozilla/5.0 (compatible; CKANCorpusBuilder/1.0; +https://example.org)"
TIMEOUT = 60

seed_df = pd.read_csv(SEED_PATH)
RESOURCES = seed_df.to_dict(orient="records")

print(f"Working directory: {Path.cwd().resolve()}")
print(f"Resolved notebook directory: {NOTEBOOK_DIR}")
print(f"Seed URL file: {SEED_PATH}")
print(f"Loaded {len(RESOURCES)} seed URLs")
print(f"Raw downloads will be saved to: {RAW_DIR}")
print(f"Cleaned text will be saved to: {CLEAN_DIR}")
print(f"Metadata will be saved to: {META_DIR}")

seed_df


Working directory: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning
Resolved notebook directory: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning
Seed URL file: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/subsidence_groundwater_seed_urls.csv
Loaded 15 seed URLs
Raw downloads will be saved to: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/raw
Cleaned text will be saved to: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/cleaned
Metadata will be saved to: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/metadata


,title,category,url,notes
0,Texas State Water Plan 2022,plan,https://www.twdb.texas.gov/waterplanning/swp/2...,Statewide planning PDF
1,Region H Water Plan 2021,plan,https://www.twdb.texas.gov/waterplanning/rwp/p...,Houston-region planning PDF
2,Harris-Galveston Subsidence District Regulator...,plan,https://hgsubsidence.org/wp-content/uploads/20...,HGSD regulatory plan PDF
3,Texas Tribune Groundwater Rights and Water Crisis,news,https://www.texastribune.org/2025/05/29/texas-...,Current Texas Tribune groundwater policy article
4,Texas Tribune Groundwater Pumping on Gulf Coas...,news,https://www.texastribune.org/2013/12/20/ground...,Texas Tribune subsidence article
5,ProPublica Climate Migration,news,https://projects.propublica.org/climate-migrat...,Broad climate piece; HTML
6,USGS Houston-Galveston Bay Area Texas From Spa...,technical,https://pubs.usgs.gov/fs/fs-110-02/pdf/FS_110-...,USGS subsidence fact sheet PDF
7,USGS Water Data for Texas,technical,https://waterdata.usgs.gov/tx/nwis/,HTML/API landing page
8,BEG Hydrogeology of Gulf Coast Aquifers Housto...,technical,https://www.beg.utexas.edu/files/publications/...,BEG Gulf Coast aquifer report PDF
9,NASA Earth Observatory Sinking Land in Houston,technical,https://earthobservatory.nasa.gov/images/14655...,HTML article page


In [2]:
def slugify(value: str) -> str:
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-")


def build_session():
    retry = Retry(
        total=3,
        read=3,
        connect=3,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    s = requests.Session()
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": USER_AGENT})
    return s


def infer_extension(url, content_type):
    path = urlparse(url).path
    suffix = Path(path).suffix.lower()
    if suffix in {".pdf", ".html", ".htm", ".csv", ".json", ".xml", ".txt", ".zip", ".tif", ".tiff"}:
        return suffix

    if content_type:
        content_type = content_type.split(";")[0].strip().lower()
        guess = mimetypes.guess_extension(content_type)
        if guess:
            if guess == ".jpe":
                return ".jpg"
            return guess

    return ".bin"


def is_html_path(path: Path) -> bool:
    return path.suffix.lower() in {".html", ".htm"}


def clean_html_text(html_text: str) -> dict[str, object]:
    soup = BeautifulSoup(html_text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "form", "iframe"]):
        tag.decompose()

    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    meta_description = ""
    meta = soup.find("meta", attrs={"name": "description"})
    if meta and meta.get("content"):
        meta_description = meta["content"].strip()

    text = soup.get_text("\n")
    lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in lines if line]
    cleaned_text = "\n".join(lines)

    lowered = cleaned_text.lower()
    is_error_page = any(
        phrase in lowered
        for phrase in [
            "404 error",
            "page not found",
            "site has moved",
            "access denied",
            "forbidden",
            "temporarily unavailable",
        ]
    )
    is_thin_page = len(cleaned_text) < 1500

    return {
        "title": title,
        "meta_description": meta_description,
        "cleaned_text": cleaned_text,
        "text_char_count": len(cleaned_text),
        "is_error_page": is_error_page,
        "is_thin_page": is_thin_page,
    }


def extract_html_to_text(saved_path: str, title: str) -> dict[str, object]:
    html_path = Path(saved_path)
    if not html_path.exists() or not is_html_path(html_path):
        return {
            "extracted_text_path": None,
            "page_title": None,
            "meta_description": None,
            "text_char_count": None,
            "is_error_page": False,
            "is_thin_page": False,
        }

    html_text = html_path.read_text(encoding="utf-8", errors="ignore")
    extracted = clean_html_text(html_text)
    text_path = CLEAN_DIR / f"{slugify(title)}.txt"
    text_path.write_text(extracted["cleaned_text"], encoding="utf-8")

    return {
        "extracted_text_path": str(text_path),
        "page_title": extracted["title"],
        "meta_description": extracted["meta_description"],
        "text_char_count": extracted["text_char_count"],
        "is_error_page": extracted["is_error_page"],
        "is_thin_page": extracted["is_thin_page"],
    }


def download_one(session, record):
    title = record["title"]
    category = record["category"]
    url = record["url"]
    notes = record.get("notes", "")
    slug = slugify(title)

    result = {
        "title": title,
        "category": category,
        "url": url,
        "notes": notes,
        "status": "failed",
        "http_status": None,
        "content_type": None,
        "saved_path": None,
        "final_url": None,
        "content_bytes": None,
        "extracted_text_path": None,
        "page_title": None,
        "meta_description": None,
        "text_char_count": None,
        "is_error_page": False,
        "is_thin_page": False,
        "error": None,
    }

    try:
        with session.get(url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
            result["http_status"] = r.status_code
            result["content_type"] = r.headers.get("content-type")
            r.raise_for_status()

            ext = infer_extension(r.url, r.headers.get("content-type", ""))
            out_path = RAW_DIR / f"{slug}{ext}"

            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 64):
                    if chunk:
                        f.write(chunk)

            result["status"] = "downloaded"
            result["saved_path"] = str(out_path)
            result["final_url"] = r.url
            result["content_bytes"] = out_path.stat().st_size

            result.update(extract_html_to_text(str(out_path), title))

    except Exception as e:
        result["error"] = repr(e)

    return result


In [3]:
session = build_session()
results = []

for i, record in enumerate(RESOURCES, start=1):
    print(f"[{i}/{len(RESOURCES)}] {record['title']}")
    res = download_one(session, record)
    results.append(res)
    print("  ->", res["status"], res.get("http_status"), res.get("saved_path") or res.get("error"))
    if res.get("extracted_text_path"):
        print("     extracted text ->", res["extracted_text_path"], f"({res['text_char_count']} chars)")
        if res.get("is_error_page"):
            print("     warning -> page looks like an error/redirect capture")
        elif res.get("is_thin_page"):
            print("     warning -> page text is thin and may be mostly boilerplate")
    time.sleep(1)

manifest = pd.DataFrame(results)
manifest_path = META_DIR / "download_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"\nSaved manifest: {manifest_path}")
print(f"Downloaded files directory: {RAW_DIR}")
print(f"Cleaned text directory: {CLEAN_DIR}")
manifest


[1/15] Texas State Water Plan 2022
  -> downloaded 200 /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/raw/texas-state-water-plan-2022.pdf
[2/15] Region H Water Plan 2021
  -> downloaded 200 /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/raw/region-h-water-plan-2021.pdf
[3/15] Harris-Galveston Subsidence District Regulatory Plan (amended 2021)
  -> downloaded 200 /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/raw/harris-galveston-subsidence-district-regulatory-plan-amended-2021.pdf
[4/15] Texas Tribune Groundwater Rights and Water Crisis
  -> downloaded 200 /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/raw/texas-tribune-groundwater-rights-and-water-crisis.html
     extracted text -> /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsid

,title,category,url,notes,status,http_status,content_type,saved_path,final_url,content_bytes,extracted_text_path,page_title,meta_description,text_char_count,is_error_page,is_thin_page,error
0,Texas State Water Plan 2022,plan,https://www.twdb.texas.gov/waterplanning/swp/2...,Statewide planning PDF,downloaded,200,text/html,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://www.twdb.texas.gov/waterplanning/swp/2...,60033,None,None,None,NaN,False,False,None
1,Region H Water Plan 2021,plan,https://www.twdb.texas.gov/waterplanning/rwp/p...,Houston-region planning PDF,downloaded,200,text/html,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://www.twdb.texas.gov/waterplanning/rwp/p...,60037,None,None,None,NaN,False,False,None
2,Harris-Galveston Subsidence District Regulator...,plan,https://hgsubsidence.org/wp-content/uploads/20...,HGSD regulatory plan PDF,downloaded,200,application/pdf,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://hgsubsidence.org/wp-content/uploads/20...,600182,None,None,None,NaN,False,False,None
3,Texas Tribune Groundwater Rights and Water Crisis,news,https://www.texastribune.org/2025/05/29/texas-...,Current Texas Tribune groundwater policy article,downloaded,200,text/html; charset=UTF-8,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://www.texastribune.org/2025/05/29/texas-...,453051,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,The one thing Texas won’t do to save its water...,Texas property owners can use nearly as much w...,68111.0,False,False,None
4,Texas Tribune Groundwater Pumping on Gulf Coas...,news,https://www.texastribune.org/2013/12/20/ground...,Texas Tribune subsidence article,downloaded,200,text/html; charset=UTF-8,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://www.texastribune.org/2013/12/20/ground...,363870,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,Groundwater Pumping on Gulf Coast Leads to Sub...,"Unlike the rest of the state, the Texas Gulf C...",31001.0,False,False,None
5,ProPublica Climate Migration,news,https://projects.propublica.org/climate-migrat...,Broad climate piece; HTML,downloaded,200,text/html,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://projects.propublica.org/climate-migrat...,38318,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,New Climate Maps Show a Transformed United Sta...,,1541.0,False,False,None
6,USGS Houston-Galveston Bay Area Texas From Spa...,technical,https://pubs.usgs.gov/fs/fs-110-02/pdf/FS_110-...,USGS subsidence fact sheet PDF,downloaded,200,application/pdf,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://pubs.usgs.gov/fs/fs-110-02/pdf/FS_110-...,7322971,None,None,None,NaN,False,False,None
7,USGS Water Data for Texas,technical,https://waterdata.usgs.gov/tx/nwis/,HTML/API landing page,downloaded,200,text/html; charset=utf-8,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://waterdata.usgs.gov/,21826,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,USGS Water Data for the Nation,Use USGS data to view water conditions near yo...,1276.0,False,True,None
8,BEG Hydrogeology of Gulf Coast Aquifers Housto...,technical,https://www.beg.utexas.edu/files/publications/...,BEG Gulf Coast aquifer report PDF,downloaded,200,application/pdf,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://www.beg.utexas.edu/files/publications/...,3444258,None,None,None,NaN,False,False,None
9,NASA Earth Observatory Sinking Land in Houston,technical,https://earthobservatory.nasa.gov/images/14655...,HTML article page,downloaded,200,text/html; charset=UTF-8,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,https://science.nasa.gov/earth/earth-observatory/,278386,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,Earth Observatory - NASA Science,"NASA's Earth Observatory brings you the Earth,...",8339.0,False,False,None


In [4]:
# Optional: create a simple CKAN-ready resource table based on what actually downloaded

manifest = pd.read_csv(META_DIR / "download_manifest.csv")
downloaded = manifest[manifest["status"] == "downloaded"].copy()
downloaded["format"] = downloaded["saved_path"].apply(lambda p: Path(p).suffix.replace(".", "").upper())
downloaded["name"] = downloaded["title"]
downloaded["description"] = downloaded["notes"]

ckan_seed = downloaded[[
    "name",
    "category",
    "format",
    "url",
    "final_url",
    "saved_path",
    "extracted_text_path",
    "description",
    "content_type",
    "page_title",
    "text_char_count",
    "is_error_page",
    "is_thin_page",
]].rename(columns={"category": "source_type"})

ckan_seed_path = META_DIR / "ckan_resource_seed.csv"
ckan_seed.to_csv(ckan_seed_path, index=False)

print("Saved CKAN seed:", ckan_seed_path)
ckan_seed


Saved CKAN seed: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/metadata/ckan_resource_seed.csv


,name,source_type,format,url,final_url,saved_path,extracted_text_path,description,content_type,page_title,text_char_count,is_error_page,is_thin_page
0,Texas State Water Plan 2022,plan,PDF,https://www.twdb.texas.gov/waterplanning/swp/2...,https://www.twdb.texas.gov/waterplanning/swp/2...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,NaN,Statewide planning PDF,text/html,NaN,NaN,False,False
1,Region H Water Plan 2021,plan,PDF,https://www.twdb.texas.gov/waterplanning/rwp/p...,https://www.twdb.texas.gov/waterplanning/rwp/p...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,NaN,Houston-region planning PDF,text/html,NaN,NaN,False,False
2,Harris-Galveston Subsidence District Regulator...,plan,PDF,https://hgsubsidence.org/wp-content/uploads/20...,https://hgsubsidence.org/wp-content/uploads/20...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,NaN,HGSD regulatory plan PDF,application/pdf,NaN,NaN,False,False
3,Texas Tribune Groundwater Rights and Water Crisis,news,HTML,https://www.texastribune.org/2025/05/29/texas-...,https://www.texastribune.org/2025/05/29/texas-...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,Current Texas Tribune groundwater policy article,text/html; charset=UTF-8,The one thing Texas won’t do to save its water...,68111.0,False,False
4,Texas Tribune Groundwater Pumping on Gulf Coas...,news,HTML,https://www.texastribune.org/2013/12/20/ground...,https://www.texastribune.org/2013/12/20/ground...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,Texas Tribune subsidence article,text/html; charset=UTF-8,Groundwater Pumping on Gulf Coast Leads to Sub...,31001.0,False,False
5,ProPublica Climate Migration,news,HTML,https://projects.propublica.org/climate-migrat...,https://projects.propublica.org/climate-migrat...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,Broad climate piece; HTML,text/html,New Climate Maps Show a Transformed United Sta...,1541.0,False,False
6,USGS Houston-Galveston Bay Area Texas From Spa...,technical,PDF,https://pubs.usgs.gov/fs/fs-110-02/pdf/FS_110-...,https://pubs.usgs.gov/fs/fs-110-02/pdf/FS_110-...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,NaN,USGS subsidence fact sheet PDF,application/pdf,NaN,NaN,False,False
7,USGS Water Data for Texas,technical,HTML,https://waterdata.usgs.gov/tx/nwis/,https://waterdata.usgs.gov/,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,HTML/API landing page,text/html; charset=utf-8,USGS Water Data for the Nation,1276.0,False,True
8,BEG Hydrogeology of Gulf Coast Aquifers Housto...,technical,PDF,https://www.beg.utexas.edu/files/publications/...,https://www.beg.utexas.edu/files/publications/...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,NaN,BEG Gulf Coast aquifer report PDF,application/pdf,NaN,NaN,False,False
9,NASA Earth Observatory Sinking Land in Houston,technical,HTML,https://earthobservatory.nasa.gov/images/14655...,https://science.nasa.gov/earth/earth-observatory/,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,/Users/wmobley/Documents/GitHub/DSO/DSO-Instit...,HTML article page,text/html; charset=UTF-8,Earth Observatory - NASA Science,8339.0,False,False


In [5]:
# Optional: save one JSON record per resource for downstream NLP / annotation work

manifest = pd.read_csv(META_DIR / "download_manifest.csv")

for row in manifest.to_dict(orient="records"):
    out_json = META_DIR / f"{slugify(row['title'])}.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2)

print(f"Wrote {len(manifest)} metadata JSON files to {META_DIR}")


Wrote 15 metadata JSON files to /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-3/Morning/data/subsidence_groundwater_corpus/metadata



## Notes

- Some publisher pages may block automated access or change URLs over time.
- For CKAN, you can ingest either:
  - the saved binary/HTML files in `raw/`, or
  - the `ckan_resource_seed.csv` file in `metadata/`
- If you want article **text extraction** next, add a cleaning step with libraries such as `trafilatura` or `readability-lxml`.
